# D69 Eval Drift Code

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

last modified date : 2026.11

# 평가와 드리프트 측정 실습

## 상황과 오늘 할 일

사내 규정 도우미를 배포하기 전에 어떤 질문에 답할 수 있고 어떤 질문에는 답하지 않아야 하는지 확인합니다. 문서에서 정답과 근거를 확인해 평가 세트를 만들고, 실제 소형 언어 모델을 사용하는 RAG 서비스에 질문을 보냅니다. Phoenix에 평가 실험을 남긴 뒤 같은 질문으로 프롬프트 변경 전후를 확인하고 실패 요청의 트레이스를 읽습니다.

## 먼저 알아둘 개념

**평가 세트**는 입력 질문, 사용자의 권한, 참조 답과 근거를 묶은 사례 모음입니다. **평가 기준**은 무엇을 맞았다고 볼지 정하는 규칙입니다. 오늘의 숫자·출처 검사는 사내 규정 조회에 한정한 기준이며, 모든 답변의 의미와 안전성을 평가하는 일반 지표는 아닙니다.

**Phoenix**는 평가 데이터셋·실험과 요청 트레이스를 함께 확인하는 도구입니다. **트레이스**는 요청 하나의 처리 흐름이고 **스팬**은 검색·생성 같은 한 단계의 기록입니다. 평가에서 실패를 찾은 뒤 트레이스로 원인을 조사합니다. **드리프트**는 운영 질문이나 자료의 분포가 달라지는 현상으로, 고정 평가만으로 놓치는 실패를 찾아 평가 세트를 보완할 때 확인합니다.

## 학습목표

- 질문·사용자·참조 답·근거를 포함한 평가 세트를 작성할 수 있다.
- 실제 RAG 응답을 평가하고 Phoenix의 데이터셋·실험 화면에서 사례별 결과를 확인할 수 있다.
- 같은 평가 세트로 프롬프트 변경을 확인하고 실패 요청의 검색·생성 트레이스를 찾아 설명할 수 있다.

## 진행 순서와 시간

| 시간 | 활동 | 분 |
|---|---|---:|
| 10:00~10:45 | 평가의 목적·참조 답·검색과 생성 오류 구분 | 45 |
| 10:45~11:05 | 휴식 | 20 |
| 11:05~12:00 | 평가 세트 작성과 RAG 서비스 기동 | 55 |
| 12:00~13:00 | 기준 평가 실행과 사례별 응답 검토 | 60 |
| 13:00~14:00 | 점심 | 60 |
| 14:00~15:20 | 프롬프트 수정과 Phoenix 실험 비교 | 80 |
| 15:20~15:40 | 휴식 | 20 |
| 15:40~16:40 | 실패 트레이스 확인·회귀 사례 작성 | 60 |
| 16:40~17:00 | 결과 저장·서비스 종료 | 20 |

![그림: 평가와 실패 조사](lms-img:0)

![공식자료 또는 실제 실행 화면](lms-img:0)

도구 소개: Phoenix 공식 홈페이지 화면. 출처: [공식자료](https://arize.com/phoenix/) (2026-11-25).

## 0. 실습 준비

Linux CPU 환경에서 실행합니다. 기본 경로는 Qwen2.5-0.5B-Instruct를 CPU로 실제 실행하며 임베딩은 multilingual-e5-small을 사용합니다. 메모리를 8GB 이상 사용할 수 있는 환경을 권장합니다. 기존에 승인된 생성 서버를 사용할 때는 `D69_GEN_URL_A`와 `D69_MODEL`을 설정합니다. 새 GPU를 대여하지 않습니다.

처음 실행할 때만 모델 파일을 내려받습니다. 내려받은 모델은 실습 폴더의 캐시에 보관합니다. 모델 로드에 실패하면 오류를 확인하며, 대본이나 정답표 응답으로 자동 대체하지 않습니다.

### 로컬 Docker에서 시작하기

컨테이너 심화와 쿠버네티스 노트(D67)에서 사용한 Docker를 켜고, 로컬에 이번 차시 전용 폴더를 만들어 이 노트북을 넣습니다. 그 폴더의 PowerShell 또는 터미널에서 아래 명령을 실행합니다. Linux·macOS에서 `python` 명령이 없으면 `python3`으로 바꿉니다.

```shell
python -c "import json; from pathlib import Path; n=json.loads(Path('D69_Eval_Drift_Code.ipynb').read_text(encoding='utf-8')); exec(''.join(next(c for c in n['cells'] if c.get('metadata',{}).get('lesson_role')=='docker_bootstrap')['source']))"
cd local_d69
docker compose up --build -d
docker compose ps
```

`healthy`가 되면 준비 코드가 출력한 토큰 포함 주소로 JupyterLab에 접속하고, 이 노트북을 열어 환경 설정 셀부터 실행합니다. 준비 코드는 Docker 실행 파일만 생성합니다. 문서·서버·평가·정책 코드는 아래 본 실습에서 작성합니다. 설치 후 커널 재시작이 필요하면 환경 설정 셀부터 다시 실행합니다. 기본 CPU 실습은 GPU를 대여하지 않습니다.

노트북과 결과는 바인드 마운트로 연결한 로컬 실습 폴더에 남습니다. JupyterLab 기본 주소는 `http://localhost:8888`이며 다른 차시의 컨테이너는 종료한 상태로 시작합니다. 접속이 안 되면 호스트 터미널에서 `docker compose ps`와 `docker compose logs --tail 40 notebook`을 확인합니다.

In [30]:
import os
import secrets
from pathlib import Path

DAY=69
NOTEBOOK='D69_Eval_Drift_Code.ipynb'
UI_PORTS=[8694]
if os.environ.get('COURSE_LOCAL_DOCKER')=='1':
    print('로컬 Docker의 JupyterLab입니다. 환경 설정 셀부터 진행하세요.')
else:
    lesson_dir=Path.cwd().resolve()
    if not (lesson_dir/NOTEBOOK).is_file():
        raise FileNotFoundError(f'{NOTEBOOK}이 있는 실습 폴더에서 실행하세요. 현재 위치: {lesson_dir}')
    local=lesson_dir/f'local_d{DAY}'
    local.mkdir(exist_ok=True)
    (local/'Dockerfile').write_text('''FROM python:3.12-slim
RUN apt-get update && apt-get install -y --no-install-recommends wget ca-certificates libgomp1 && rm -rf /var/lib/apt/lists/*
RUN pip install --no-cache-dir "jupyterlab==4.6.6"
WORKDIR /workspace
''',encoding='utf8')
    mapping='\n'.join(f'      - "127.0.0.1:{p}:{p}"' for p in [8888]+UI_PORTS)
    compose=f'''name: course-d{DAY}
services:
  notebook:
    build: .
    platform: linux/amd64
    user: "${{COURSE_UID:-1000}}:0"
    init: true
    working_dir: /workspace
    environment:
      COURSE_LOCAL_DOCKER: "1"
      USER: student
      LOGNAME: student
      HOME: /tmp/course-home
      PATH: /tmp/course-home/.local/bin:/usr/local/bin:/usr/local/sbin:/usr/sbin:/usr/bin:/sbin:/bin
      JUPYTER_TOKEN: "${{COURSE_TOKEN:?준비 코드를 먼저 실행하세요}}"
      JUPYTER_RUNTIME_DIR: /tmp/course-jupyter-runtime
      JUPYTER_DATA_DIR: /tmp/course-jupyter-data
      JUPYTER_CONFIG_DIR: /tmp/course-jupyter-config
      IPYTHONDIR: /tmp/course-ipython
      HF_HOME: /workspace/.cache/huggingface
      HF_HUB_DISABLE_TELEMETRY: "1"
      PYTHONUNBUFFERED: "1"
    ports:
{mapping}
    volumes:
      - type: bind
        source: ..
        target: /workspace
    command: ["python", "-m", "jupyterlab", "--ip=0.0.0.0", "--port=8888", "--no-browser", "--ServerApp.port_retries=0", "--ServerApp.root_dir=/workspace", "--ServerApp.use_redirect_file=False"]
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8888/login', timeout=5)"]
      interval: 10s
      timeout: 6s
      retries: 12
      start_period: 20s
'''
    (local/'compose.yaml').write_text(compose,encoding='utf8')
    (local/'.dockerignore').write_text('*\n!Dockerfile\n',encoding='utf8')
    env=local/'.env'
    if not env.exists():
        uid=os.getuid() if hasattr(os,'getuid') else 1000
        env.write_text(f'COURSE_UID={uid}\nCOURSE_TOKEN={secrets.token_urlsafe(24)}\n',encoding='utf8')
    values=dict(line.split('=',1) for line in env.read_text(encoding='utf8').splitlines() if '=' in line)
    print('준비 파일:',local)
    print(f'cd local_d{DAY}\ndocker compose up --build -d\ndocker compose ps')
    print(f'http://localhost:8888/lab/tree/{NOTEBOOK}?token={values["COURSE_TOKEN"]}')

로컬 Docker의 JupyterLab입니다. 환경 설정 셀부터 진행하세요.


In [31]:
from pathlib import Path
import os, sys, json, subprocess
HAS_GPU = False
if "COURSE_DIR" not in globals():
    COURSE_DIR = Path.cwd().resolve()
WORK = COURSE_DIR / "work_d69"
WORK.mkdir(parents=True, exist_ok=True)
os.environ["D69_WORK"] = str(WORK)
os.chdir(WORK)
print("작업 폴더:", WORK)

작업 폴더: /workspace/work_d69


In [32]:
def save_submission(decision, **evidence):
    if not isinstance(decision, str) or not decision.strip():
        raise ValueError("관찰한 근거와 최종 판단을 작성하십시오.")
    def serialize(value):
        if isinstance(value, Path): return str(value)
        if hasattr(value, "item"): return value.item()
        raise TypeError(type(value).__name__)
    target = WORK / "d69_submission.json"
    target.write_text(json.dumps({"day":69,"decision":decision,"evidence":evidence},
        ensure_ascii=False,indent=2,default=serialize),encoding="utf-8")
    print("제출 파일:", target)

def close_services():
    scope = globals()
    if scope.get("TRAFFIC") is not None:
        scope["TRAFFIC"].stop()
    if callable(scope.get("stop_all")):
        scope["stop_all"]()
    for name in ["PROCS","procs","PROCESSES","processes","SERVERS"]:
        for value in (scope.get(name) or {}).values():
            proc = value.get("proc",value.get("p")) if isinstance(value,dict) else value
            if hasattr(proc,"poll") and proc.poll() is None:
                proc.terminate()
                try:proc.wait(timeout=10)
                except subprocess.TimeoutExpired:proc.kill()
    for name in ["HTTP","_mcp_log","qc"]:
        resource=scope.get(name)
        if resource is not None and callable(getattr(resource,"close",None)):
            resource.close()
    print("실습 서버 종료 처리를 마쳤습니다.")

In [33]:
%pip install -q torch --index-url https://download.pytorch.org/whl/cpu
%pip install -q "transformers>=5.17,<6" "sentence-transformers>=6,<7" "accelerate>=1.15,<2" "arize-phoenix>=20,<21" "opentelemetry-exporter-otlp-proto-http>=1.45,<2" "fastapi>=0.141,<0.142" "uvicorn[standard]>=0.53,<0.54" "httpx>=0.28,<1" "pandas>=2,<3"


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. 평가 세트와 채점 기준 작성

제공된 규정은 가상 회사의 합성 문서입니다. 정답은 모델의 답에서 가져오지 않고 문서에서 확인합니다. 일반 조회·계산·권한상 답할 수 없는 사례를 함께 넣고, 참조 답이 없는 이유도 적습니다.

In [34]:
import json, os, random, re, subprocess, sys, time
from collections import Counter, defaultdict
from importlib.metadata import version
from pathlib import Path
import httpx
import numpy as np
import torch
import pandas as pd

GEN = "cpu"
PORTS={name:8690+i for i,name in enumerate(["gen-a","gen-b","rag-server","gateway","phoenix","phoenix-grpc"])}
WORK=Path(os.environ["D69_WORK"])
for folder in ["spans","logs","content"]:(WORK/folder).mkdir(exist_ok=True)
os.environ.setdefault("HF_HOME",str(WORK/"model_cache"))
os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
os.environ["OMP_NUM_THREADS"]="4"
os.environ["MKL_NUM_THREADS"]="4"
sys.path.insert(0,str(WORK))
print("실행 환경: CPU / 생성 모델:",os.environ.get("D69_MODEL","Qwen/Qwen2.5-0.5B-Instruct"))
print({name:version(name) for name in ["torch","transformers","sentence-transformers","arize-phoenix"]})

실행 환경: CPU / 생성 모델: Qwen/Qwen2.5-0.5B-Instruct
{'torch': '2.15.0+cpu', 'transformers': '5.24.1', 'sentence-transformers': '6.3.0', 'arize-phoenix': '20.24.0'}


In [35]:
DOCS = [{"id": i, "부서": i.split("-")[0], "제목": t, "본문": b} for i, t, b in [
    ("공개-01", "근무 시간", "기본 근무 시간은 오전 9시부터 오후 6시까지이다."),
    ("공개-02", "재택근무", "재택근무는 주 2일까지 신청할 수 있다. 신청은 사용 전날 오후 3시까지 팀장에게 한다. 신입 직원은 입사 후 3개월이 지나야 재택근무를 신청할 수 있다."),
    ("공개-03", "연차 휴가", "1년 이상 근무한 직원에게는 연 15일의 연차를 준다. 연차는 반일 단위로 나누어 쓸 수 있다."),
    ("공개-04", "반차", "반차는 하루 근무 시간의 절반인 4시간으로 계산하며 연차 0.5일을 쓴 것으로 본다. 오전 반차는 오후 2시에 출근하고 오후 반차는 오후 1시에 퇴근한다."),
    ("공개-05", "국내 출장비", "국내 출장의 숙박비는 1박에 10만 원까지 실비로 지급한다. 식비는 하루 3만 원을 정액으로 지급한다."),
    ("공개-06", "해외 출장비", "해외 출장의 숙박비는 1박에 20만 원까지 실비로 지급한다. 일비는 하루 5만 원을 정액으로 지급한다. 귀국한 뒤 10일 안에 정산을 신청한다."),
    ("공개-07", "정보 보안", "회사 노트북은 자리를 비울 때 반드시 화면을 잠근다. 고객 정보가 담긴 파일은 외부 메일로 보내지 않는다. 보안 사고를 알게 된 직원은 1시간 안에 보안팀에 알린다."),
    ("공개-08", "경조 휴가", "본인 결혼 시 5일, 자녀 결혼 시 1일의 경조 휴가를 준다. 부모가 사망한 경우에는 5일을 준다."),
    ("공개-09", "교육 지원", "직무와 관련된 외부 교육비는 연 100만 원까지 지원한다. 교육을 마친 직원은 2주 안에 결과 보고서를 낸다."),
    ("공개-10", "장비 대여", "업무용 노트북은 4년마다 바꿔 준다."),
    ("공개-11", "법인카드 사용", "법인카드는 업무 목적에만 쓴다. 한 번에 50만 원을 넘는 결제는 재무팀의 사전 승인을 받는다."),
    ("공개-14", "건강검진", "건강검진은 2년에 1번 받는다. 만 40세 이상 직원은 해마다 받는다."),
    ("공개-15", "육아휴직", "육아휴직은 자녀 한 명당 최대 1년까지 쓸 수 있다. 휴직을 시작하기 30일 전까지 인사팀에 신청한다."),
    ("공개-16", "주차 지원", "회사 건물 주차장은 임원과 장애인 직원에게 먼저 배정한다. 나머지 직원에게는 월 주차비를 7만 원까지 지원한다."),
    ("공개-17", "동호회 지원", "회원이 5명 이상인 사내 동호회에는 회원 1명당 한 달에 2만 원을 지원한다."),
    ("인사-01", "평가 등급", "평가 등급은 S, A, B, C 네 단계이다. S 등급은 팀별 상위 10퍼센트로 제한한다."),
    ("인사-02", "성과급", "S 등급의 성과급은 기본급의 200퍼센트이고 A 등급은 150퍼센트이다."),
    ("재무-01", "법인카드 한도", "법인카드의 월 한도는 300만 원이다. 한도를 올리려면 재무팀장의 승인을 받는다. 한도 상향은 최대 3개월 동안 유지된다."),
    ("재무-02", "예산 전용", "분기 예산의 10퍼센트 안에서는 팀장 승인으로 다른 항목에 옮겨 쓸 수 있다."),
    ("영업-01", "4분기 영업 목표", "4분기 목표는 3분기 실적의 120퍼센트이다. 신규 계약 비중은 절반 이상으로 한다."),
    ("보안-01", "보안 사고 등급", "1등급 보안 사고는 30분 안에 정보보호팀장에게 보고한다. 2등급 사고는 4시간 안에 보고한다.")]]

# 생산과 품질은 지금 기록에만 나오는 새 부서입니다. 그 부서의 규정은 아직 인덱스에 없습니다
USERS = {"김총무": {"부서": "총무", "열람": ["공개"]}, "정개발": {"부서": "개발", "열람": ["공개"]},
         "박인사": {"부서": "인사", "열람": ["공개", "인사"]}, "이영업": {"부서": "영업", "열람": ["공개", "영업"]},
         "최재무": {"부서": "재무", "열람": ["공개", "재무"]}, "한생산": {"부서": "생산", "열람": ["공개"]},
         "오품질": {"부서": "품질", "열람": ["공개"]}}
print(f"문서 {len(DOCS)}건 :", dict(Counter(d["부서"] for d in DOCS)))

문서 21건 : {'공개': 15, '인사': 2, '재무': 2, '영업': 1, '보안': 1}


In [36]:
QUESTIONS = [dict(zip(["id", "user", "q", "answer", "근거", "kind"], x)) for x in [
    ("Q01", "김총무", "재택근무는 한 주에 최대 며칠까지 신청할 수 있습니까?", 2, ["공개-02"], "조회"),
    ("Q02", "김총무", "재택근무 신청은 전날 오후 몇 시까지 해야 합니까?", 3, ["공개-02"], "조회"),
    ("Q03", "정개발", "1년 이상 근무한 직원의 연차는 1년에 며칠입니까?", 15, ["공개-03"], "조회"),
    ("Q04", "정개발", "반차는 몇 시간으로 계산합니까?", 4, ["공개-04"], "조회"),
    ("Q05", "김총무", "국내 출장의 1박 숙박비는 최대 몇 만 원입니까?", 10, ["공개-05"], "조회"),
    ("Q06", "이영업", "해외 출장의 1박 숙박비는 최대 몇 만 원입니까?", 20, ["공개-06"], "조회"),
    ("Q07", "정개발", "보안 사고를 알게 되면 몇 시간 안에 알려야 합니까?", 1, ["공개-07"], "조회"),
    ("Q08", "김총무", "본인 결혼 때 경조 휴가는 며칠입니까?", 5, ["공개-08"], "조회"),
    ("Q09", "최재무", "자녀 결혼 때 경조 휴가는 며칠입니까?", 1, ["공개-08"], "조회"),
    ("Q10", "정개발", "외부 교육비는 1년에 최대 몇 만 원까지 지원됩니까?", 100, ["공개-09"], "조회"),
    ("Q11", "이영업", "법인카드로 한 번에 몇 만 원을 넘게 결제하면 사전 승인이 필요합니까?", 50, ["공개-11"], "조회"),
    ("Q12", "박인사", "육아휴직은 시작하기 며칠 전까지 신청해야 합니까?", 30, ["공개-15"], "조회"),
    ("Q13", "김총무", "건강검진은 몇 년에 한 번 받습니까?", 2, ["공개-14"], "조회"),
    ("Q14", "이영업", "국내 출장을 3박 4일로 다녀오면 숙박비 상한과 식비를 합해 최대 몇 만 원을 받습니까?", 42, ["공개-05"], "계산"),
    ("Q15", "정개발", "연차 15일 가운데 반차를 3번 썼다면 남은 연차는 며칠입니까?", 13.5, ["공개-03", "공개-04"], "계산"),
    ("Q18", "박인사", "S 등급은 팀별 상위 몇 퍼센트입니까?", 10, ["인사-01"], "권한"),
    ("Q19", "이영업", "S 등급은 팀별 상위 몇 퍼센트입니까?", None, [], "권한"),
    ("Q20", "최재무", "법인카드의 월 한도는 몇 만 원입니까?", 300, ["재무-01"], "권한"),
    ("Q21", "김총무", "법인카드의 월 한도는 몇 만 원입니까?", None, [], "권한"),
    ("Q24", "정개발", "주차비는 한 달에 몇 만 원까지 지원됩니까?", 7, ["공개-16"], "조회")]]
QMAP = {x["id"]: x for x in QUESTIONS}

# 같은 뜻을 다른 말투로 물은 질문. 재질문에도 이 말투를 씁니다
PARA = {"Q01": ["재택 일주일에 며칠까지 돼요?", "재택근무 주 며칠 가능한가요?"],
        "Q02": ["재택 신청 마감이 전날 몇 시예요?", "재택근무는 전날 언제까지 신청하나요?"],
        "Q03": ["연차 1년에 며칠 나와요?", "1년 넘게 일하면 연차가 며칠인가요?"],
        "Q04": ["반차는 몇 시간짜리예요?", "반차 한 번이 몇 시간으로 잡히나요?"],
        "Q05": ["국내 출장 호텔비 1박 상한이 얼마예요?", "국내 출장 숙박비 하루 최대 몇 만 원인가요?"],
        "Q06": ["해외 출장 숙박비 1박에 얼마까지 돼요?", "해외 출장 호텔 하루 상한이 몇 만 원이죠?"],
        "Q07": ["보안 사고 나면 몇 시간 안에 신고해요?", "보안 사고는 몇 시간 안에 보안팀에 말해야 하나요?"],
        "Q24": ["주차비 한 달 지원 얼마예요?", "월 주차 지원이 몇 만 원까지죠?"]}
print(f"질문 {len(QUESTIONS)}개 :", dict(Counter(x["kind"] for x in QUESTIONS)), f"/ 말투를 바꾼 질문 {sum(map(len, PARA.values()))}개")

질문 20개 : {'조회': 14, '계산': 2, '권한': 4} / 말투를 바꾼 질문 16개


In [37]:
# 세 종류를 포함한 기본 예시입니다. 문서를 읽고 두 사례를 더 추가해 보세요.
EVAL_IDS=["Q01","Q02","Q05","Q07","Q14","Q18","Q19","Q21"]
EVAL_CASES=[dict(QMAP[qid]) for qid in EVAL_IDS]
for row in EVAL_CASES:
    row["답이 없는 이유"]="열람 권한이 없어 답하지 않아야 함" if row["answer"] is None else ""
evaluation_file=WORK/"evaluation_cases.jsonl"
evaluation_file.write_text("\n".join(json.dumps(x,ensure_ascii=False) for x in EVAL_CASES)+"\n",encoding="utf-8")
display(pd.DataFrame(EVAL_CASES)[["id","user","q","answer","근거","답이 없는 이유"]])
print("저장:",evaluation_file)

,id,user,q,answer,근거,답이 없는 이유
0,Q01,김총무,재택근무는 한 주에 최대 며칠까지 신청할 수 있습니까?,2.0,[공개-02],
1,Q02,김총무,재택근무 신청은 전날 오후 몇 시까지 해야 합니까?,3.0,[공개-02],
2,Q05,김총무,국내 출장의 1박 숙박비는 최대 몇 만 원입니까?,10.0,[공개-05],
3,Q07,정개발,보안 사고를 알게 되면 몇 시간 안에 알려야 합니까?,1.0,[공개-07],
4,Q14,이영업,국내 출장을 3박 4일로 다녀오면 숙박비 상한과 식비를 합해 최대 몇 만 원을 받습니까?,42.0,[공개-05],
5,Q18,박인사,S 등급은 팀별 상위 몇 퍼센트입니까?,10.0,[인사-01],
6,Q19,이영업,S 등급은 팀별 상위 몇 퍼센트입니까?,NaN,[],열람 권한이 없어 답하지 않아야 함
7,Q21,김총무,법인카드의 월 한도는 몇 만 원입니까?,NaN,[],열람 권한이 없어 답하지 않아야 함


저장: /workspace/work_d69/evaluation_cases.jsonl


### 자동 평가 기준

답의 첫 숫자와 참조 숫자가 같은지, 필요한 문서가 검색됐는지, 답변에 필요한 출처가 표시됐는지를 따로 검사합니다. 권한상 답할 수 없는 사례는 정해 둔 거절 응답인지 확인합니다. 자동 판정이 틀린 사례는 원문을 읽고 수동 검토에 남깁니다. 숫자 하나가 맞았다는 이유만으로 답변 전체가 맞았다고 판단하지 않습니다.

In [38]:
REFUSAL = "근거를 찾지 못했습니다"

def grade(answer, expect):
    """사내 규정 조회용 숫자·거절 검사. 의미 검토를 대체하지 않습니다."""
    if expect is None or REFUSAL in answer:
        return expect is None and REFUSAL in answer
    clean = re.sub(r"\[[^\]]+\]", "", answer)
    clean = re.sub(r"[가-힣]+-\d+", "", clean)
    match = re.search(r"-?\d+(?:\.\d+)?", clean.replace(",", ""))
    return match is not None and abs(float(match.group()) - float(expect)) < 1e-6

print([grade(a, e) for a, e in [("답: 2. 근거: 공개-02", 2),
    ("근거: [공개-02] 전날 오후 3시까지 신청합니다.", 3),
    ("답: 7", 2), ("근거를 찾지 못했습니다.", None)]])

[True, True, False, True]


In [39]:
def answer_correct(output, expected):
    if output.get("status")!=200:
        return {"label":"실행 오류", "explanation":str(output.get("error",output.get("status")))}
    ok=grade(output.get("answer",""),expected["answer"])
    return {"score":float(ok),"label":"기준 충족" if ok else "검토 필요",
            "explanation":"사내 규정 조회용 숫자·거절 기준. 의미 전체를 보증하지 않음"}

def retrieval_hit(output, expected):
    required=set(expected["sources"])
    if not required:return {"label":"해당 없음"}
    return float(required.issubset(set(output.get("sources",[]))))

def citation_present(output, expected):
    required=set(expected["sources"])
    if not required:return {"label":"해당 없음"}
    cited=set(re.findall(r"[가-힣]+-\d+",output.get("answer","")))
    return float(required.issubset(cited))

## 2. 실제 RAG 서비스와 Phoenix 기동

아래 소스를 저장하고 생성 서버·검색 서버·게이트웨이를 순서대로 실행합니다. 생성 서버는 평가 정답을 읽지 않습니다. 임베딩으로 검색한 문서만 생성 모델에 전달합니다. 계측 코드는 요청의 단계를 연결하되 문서와 질문 원문 전체를 트레이스 속성에 넣지 않습니다.

In [40]:
%%writefile d69_otel.py
"""세 서비스가 함께 쓰는 추적 설정.

스팬은 서비스마다 spans/<서비스>.jsonl 에 한 줄씩 남깁니다.
D69_OTLP 에 주소가 있으면 그 주소(OTLP/HTTP)로도 보냅니다.
"""
import os

from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry.trace.propagation.tracecontext import TraceContextTextMapPropagator

PROP = TraceContextTextMapPropagator()       # W3C traceparent 를 읽고 씁니다


def setup(service, folder):
    provider = TracerProvider(resource=Resource.create({"service.name": service}))
    out = open(os.path.join(folder, f"{service}.jsonl"), "a", encoding="utf-8", buffering=1)
    provider.add_span_processor(SimpleSpanProcessor(
        ConsoleSpanExporter(out=out, formatter=lambda span: span.to_json(indent=None) + "\n")))
    otlp = os.environ.get("D69_OTLP")
    if otlp:
        from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
        provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(endpoint=otlp)))
    trace.set_tracer_provider(provider)
    return trace.get_tracer(service), provider

Overwriting d69_otel.py


In [41]:
%%writefile d69_embeddings.py
import os
from sentence_transformers import SentenceTransformer

def load_embedder():
    name=os.environ.get("D69_EMBED_MODEL","intfloat/multilingual-e5-small")
    model=SentenceTransformer(name,device="cpu")
    return model,name

Overwriting d69_embeddings.py


In [42]:
%%writefile gen_server.py
"""CPU에서 실제 언어 모델을 실행하는 OpenAI 호환 생성 서버."""
import itertools, os, threading
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from fastapi import FastAPI, Request
from opentelemetry import trace
from d69_otel import PROP, setup

torch.set_num_threads(4)
MODEL_ID=os.environ.get("D69_GEN_MODEL","Qwen/Qwen2.5-0.5B-Instruct")
TOK=AutoTokenizer.from_pretrained(MODEL_ID)
MODEL=AutoModelForCausalLM.from_pretrained(MODEL_ID,dtype=torch.float32).to("cpu")
tracer,provider=setup("generation",os.environ["D69_SPANS"])
LOCK=threading.Lock()
IDS=itertools.count()
app=FastAPI()

@app.get("/health")
def health():return {"status":"healthy","model":MODEL_ID,"mode":"real-cpu"}

@app.post("/v1/chat/completions")
def chat(body:dict,request:Request):
    ctx=PROP.extract(dict(request.headers))
    with tracer.start_as_current_span("POST /v1/chat/completions",context=ctx,kind=trace.SpanKind.SERVER) as span:
        prompt=TOK.apply_chat_template(body["messages"],tokenize=False,add_generation_prompt=True)
        batch=TOK(prompt,return_tensors="pt")
        limit=min(int(body.get("max_tokens",64)),80)
        with LOCK,torch.inference_mode():
            generated=MODEL.generate(**batch,max_new_tokens=limit,do_sample=False,pad_token_id=TOK.eos_token_id)
        new=generated[0,batch["input_ids"].shape[1]:]
        answer=TOK.decode(new,skip_special_tokens=True).strip()
        n_in,n_out=int(batch["input_ids"].shape[1]),int(len(new))
        span.set_attribute("gen_ai.response.model",MODEL_ID)
        span.set_attribute("gen_ai.usage.input_tokens",n_in)
        span.set_attribute("gen_ai.usage.output_tokens",n_out)
    return {"model":MODEL_ID,"choices":[{"message":{"role":"assistant","content":answer},"finish_reason":"stop" if n_out<limit else "length"}],
            "usage":{"prompt_tokens":n_in,"completion_tokens":n_out}}

Overwriting gen_server.py


In [43]:
%%writefile rag_server.py
"""RAG 서버. 임베딩 → 열람 범위 안에서 검색 → 생성 서버 호출.

스팬 : POST /ask(SERVER) 아래에 embeddings, retrieval, chat(CLIENT)
질문과 답의 원문은 스팬에 넣지 않고 content/rag.jsonl 에 따로 남깁니다
"""
import json
import os
import time

import httpx
import numpy as np
from fastapi import FastAPI, Request
from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode

from d69_otel import PROP, setup
from d69_embeddings import load_embedder

tracer, provider = setup("rag-server", os.environ["D69_SPANS"])
DOCS = json.load(open(os.environ["D69_DOCS"], encoding="utf-8"))
USERS = json.load(open(os.environ["D69_USERS"], encoding="utf-8"))
GEN_URL = os.environ["D69_GEN_URL"]
GEN_MODEL = os.environ.get("D69_GEN_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
TOP_K = int(os.environ.get("D69_TOP_K", "3"))
PROMPT_VERSION = os.environ.get("D69_PROMPT_VERSION", "v3")
SYSTEM = os.environ.get("D69_SYSTEM_PROMPT") or ("당신은 한결물산의 사내 규정 도우미입니다. 주어진 문서에 있는 내용으로만 답합니다. "
          "첫 줄은 '답 : 숫자' 꼴로 쓰고 둘째 줄에 '근거 : 문서 번호'를 씁니다. "
          "문서에 답이 없으면 '근거를 찾지 못했습니다.'라고만 씁니다. 답변은 반드시 한국어로만 작성하세요.")

EMB, EMB_NAME = load_embedder()
DIM = EMB.get_embedding_dimension()
DOC_VECS = EMB.encode(["passage: " + d["제목"] + " " + d["본문"] for d in DOCS], normalize_embeddings=True)
CLIENT = httpx.Client(timeout=float(os.environ.get("D69_GEN_TIMEOUT", "120")))
CONTENT = open(os.path.join(os.environ["D69_CONTENT"], "rag.jsonl"), "a", encoding="utf-8", buffering=1)
app = FastAPI()


@app.get("/health")
def health():
    return {"status": "healthy", "embedding": EMB_NAME, "generator": GEN_MODEL, "prompt": PROMPT_VERSION}


@app.post("/ask")
def ask(body: dict, request: Request):
    ctx = PROP.extract(dict(request.headers))
    with tracer.start_as_current_span("POST /ask", context=ctx, kind=trace.SpanKind.SERVER) as root:
        user, question = body.get("user", "-"), body.get("question", "")
        allowed = USERS.get(user, {"열람": ["공개"]})["열람"]

        with tracer.start_as_current_span(f"embeddings {EMB_NAME}") as span:
            span.set_attribute("gen_ai.operation.name", "embeddings")
            span.set_attribute("gen_ai.request.model", EMB_NAME)
            span.set_attribute("gen_ai.embeddings.dimension.count", DIM)
            qv = EMB.encode(["query: " + question], normalize_embeddings=True)[0]

        with tracer.start_as_current_span("retrieval hangyeol-rules") as span:
            span.set_attribute("gen_ai.operation.name", "retrieval")
            span.set_attribute("gen_ai.data_source.id", "hangyeol-rules")
            span.set_attribute("gen_ai.retrieval.top_k", TOP_K)
            scores = DOC_VECS @ qv
            order = [int(i) for i in np.argsort(-scores) if DOCS[int(i)]["부서"] in allowed][:TOP_K]
            hits = [DOCS[i] for i in order]
            top = float(scores[order[0]]) if order else 0.0
            span.set_attribute("app.retrieval.doc_ids", [d["id"] for d in hits])
            span.set_attribute("app.retrieval.top_score", round(top, 4))

        prompt = "\n".join(f"[{d['id']}] {d['제목']} : {d['본문']}" for d in hits) + f"\n\n질문 : {question}"
        messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
        answer, served, finish, usage = "", "-", "error", {}
        with tracer.start_as_current_span(f"chat {GEN_MODEL}", kind=trace.SpanKind.CLIENT) as span:
            for k, v in {"gen_ai.operation.name": "chat", "gen_ai.provider.name": "internal",
                         "gen_ai.request.model": GEN_MODEL, "gen_ai.request.max_tokens": 64,
                         "gen_ai.request.temperature": 0.0, "gen_ai.prompt.name": "rules-qa",
                         "gen_ai.prompt.version": PROMPT_VERSION,
                         "server.address": "127.0.0.1", "server.port": int(GEN_URL.rsplit(":", 1)[1])}.items():
                span.set_attribute(k, v)
            headers = {}
            PROP.inject(headers)                      # 생성 서버에 traceparent 를 넘깁니다
            try:
                r = CLIENT.post(GEN_URL + "/v1/chat/completions", headers=headers,
                                json={"model": GEN_MODEL, "messages": messages,
                                      "max_tokens": 64, "temperature": 0.0})
                r.raise_for_status()
                data = r.json()
                answer = data["choices"][0]["message"]["content"]
                served, finish, usage = data.get("model", "-"), data["choices"][0]["finish_reason"], data.get("usage", {})
                span.set_attribute("gen_ai.response.model", served)
                span.set_attribute("gen_ai.response.finish_reasons", [finish])
                span.set_attribute("gen_ai.usage.input_tokens", int(usage.get("prompt_tokens", 0)))
                span.set_attribute("gen_ai.usage.output_tokens", int(usage.get("completion_tokens", 0)))
            except (httpx.HTTPError, KeyError, ValueError) as e:     # 실패도 기록하고 답은 문자열로
                span.set_status(Status(StatusCode.ERROR))
                span.set_attribute("error.type", type(e).__name__)
                answer = "생성 서버에 연결되지 않았습니다."

        trace_id = format(root.get_span_context().trace_id, "032x")
        CONTENT.write(json.dumps({"trace_id": trace_id, "t": time.time(), "user": user, "question": question,
                                  "answer": answer, "sources": [d["id"] for d in hits]},
                                 ensure_ascii=False) + "\n")
        return {"answer": answer, "sources": [d["id"] for d in hits], "top_score": round(top, 4),
                "model": served, "finish": finish, "prompt_version": PROMPT_VERSION,
                "embedding": EMB_NAME, "output_tokens": int(usage.get("completion_tokens", 0))}

Overwriting rag_server.py


In [44]:
%%writefile gateway.py
"""게이트웨이. traceparent 를 이어 받거나 새로 시작하고 X-Request-ID 로 trace ID를 반환합니다.
/v1/feedback 은 사용자 신호를 trace ID와 함께 받습니다"""
import json
import os
import re
import time

import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from opentelemetry import trace

from d69_otel import PROP, setup

tracer, provider = setup("gateway", os.environ["D69_SPANS"])
RAG_URL = os.environ["D69_RAG_URL"]
CLIENT = httpx.Client(timeout=float(os.environ.get("D69_GEN_TIMEOUT", "120")) + 10)
SIGNALS = open(os.path.join(os.environ["D69_CONTENT"], "signals.jsonl"), "a", encoding="utf-8", buffering=1)
TRACE_ID = re.compile(r"^[0-9a-f]{32}$")
app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.post("/v1/ask")
def ask(body: dict, request: Request):
    incoming = PROP.extract(dict(request.headers))
    continued = trace.get_current_span(incoming).get_span_context().is_valid
    # 전달받은 컨텍스트가 없으면 서버가 이미 만든 현재 스팬을 이어받습니다.
    ctx = incoming if continued else None
    with tracer.start_as_current_span("POST /v1/ask", context=ctx, kind=trace.SpanKind.SERVER) as span:
        span.set_attribute("app.caller", body.get("user", "-"))
        span.set_attribute("app.trace.continued", continued)
        headers = {}
        PROP.inject(headers)
        try:
            r = CLIENT.post(RAG_URL + "/ask", json=body, headers=headers)
            status, data = r.status_code, r.json()
        except (httpx.HTTPError, ValueError) as e:
            status, data = 502, {"error": f"서버에 연결되지 않습니다 ({type(e).__name__})"}
        span.set_attribute("http.response.status_code", status)
        rid = format(span.get_span_context().trace_id, "032x")
    return JSONResponse(data, status_code=status, headers={"X-Request-ID": rid})


@app.post("/v1/feedback")
def feedback(body: dict):
    # 어느 답에 대한 신호인지 모르면 쓸 수 없으므로 trace ID가 없는 신호는 받지 않습니다
    if not TRACE_ID.match(str(body.get("trace_id", ""))):
        return JSONResponse({"error": "trace_id(32자)가 필요합니다"}, status_code=400)
    SIGNALS.write(json.dumps({"trace_id": body["trace_id"], "user": body.get("user", "-"),
                              "signal": body.get("signal", "-"), "t": body.get("t", time.time())},
                             ensure_ascii=False) + "\n")
    return {"ok": True}

Overwriting gateway.py


In [45]:
(WORK/"docs.json").write_text(json.dumps(DOCS,ensure_ascii=False),encoding="utf-8")
(WORK/"users.json").write_text(json.dumps(USERS,ensure_ascii=False),encoding="utf-8")
print("검색 대상 문서 수:",len(DOCS))

검색 대상 문서 수: 21


In [46]:
PROCS = {}
GEN_MODELS = {"gen-a": os.environ.get("D69_MODEL","Qwen/Qwen2.5-0.5B-Instruct")}
COMMON = {"D69_SPANS": str(WORK / "spans"), "D69_CONTENT": str(WORK / "content"), "D69_DOCS": str(WORK / "docs.json"),
          "D69_USERS": str(WORK / "users.json"),
          "PYTHONPATH": os.pathsep.join([str(WORK), os.environ.get("PYTHONPATH", "")])}
url = lambda name: f"http://127.0.0.1:{PORTS[name]}"


def alive(base):
    try:
        return httpx.get(base + "/health", timeout=2).status_code == 200
    except httpx.HTTPError:
        return False


def start(name, module, env=None):
    """서비스를 기동하고 /health 가 응답할 때까지(최대 15분) 기다립니다"""
    if name in PROCS and PROCS[name].poll() is None:
        stop(name)
    log = WORK / "logs" / f"{name}.log"
    PROCS[name] = subprocess.Popen([sys.executable, "-m", "uvicorn", f"{module}:app", "--port", str(PORTS[name])],
                                   env={**os.environ, **COMMON, **(env or {})}, stdout=open(log, "a"), stderr=subprocess.STDOUT)
    for _ in range(900):
        if PROCS[name].poll() is not None or alive(url(name)):
            break
        time.sleep(1)
    return f"{name} " + (f"준비됨 {url(name)}" if alive(url(name)) else f"기동 실패. 로그 : {log}")


def stop(name):
    if name in PROCS:
        p = PROCS.pop(name)
        p.terminate()                        # 끝낼 때 남은 스팬을 파일에 씁니다
        p.wait(timeout=30)
    return f"{name} 종료"

In [47]:
import shutil

PHOENIX_BIN = os.environ.get("D69_PHOENIX_BIN") or shutil.which("phoenix") or str(Path(sys.executable).with_name("phoenix"))
PHX = f"http://127.0.0.1:{PORTS['phoenix']}"
PHX_ENV = {"PHOENIX_PORT": str(PORTS["phoenix"]), "PHOENIX_GRPC_PORT": str(PORTS["phoenix-grpc"]),
           "PHOENIX_HOST": "0.0.0.0", "PHOENIX_WORKING_DIR": str(WORK / "phoenix"),
           "PHOENIX_ALLOW_EXTERNAL_RESOURCES": "false", "PHOENIX_TELEMETRY_ENABLED": "false"}


def start_phoenix(wait=180):
    existing=PROCS.get("phoenix")
    if existing is not None and existing.poll() is None:
        return f"Phoenix 실행 중: {PHX}"
    if PHOENIX_BIN is None:
        return "phoenix 명령이 없습니다. 설치 셀을 먼저 실행하세요"
    (WORK / "phoenix").mkdir(exist_ok=True)
    log = open(WORK / "logs" / "phoenix.log", "a")
    PROCS["phoenix"] = subprocess.Popen([PHOENIX_BIN, "serve"], env={**os.environ, **PHX_ENV},
                                        stdout=log, stderr=subprocess.STDOUT)
    for _ in range(wait):
        if PROCS["phoenix"].poll() is not None:
            return f"Phoenix 기동 실패. 로그 : {WORK / 'logs' / 'phoenix.log'}"
        try:
            if httpx.get(PHX + "/healthz", timeout=2).status_code == 200:
                return f"Phoenix 준비됨 {PHX} (OTLP/HTTP {PHX}/v1/traces)"
        except httpx.HTTPError:
            pass
        time.sleep(1)
    return "Phoenix 가 제시간에 준비되지 않았습니다"


print(start_phoenix())
if httpx.get(PHX + "/healthz",timeout=3).status_code != 200:
    raise RuntimeError("Phoenix 기동 로그를 확인한 뒤 다음 셀을 실행하십시오.")
os.environ["D69_OTLP"] = PHX + "/v1/traces"

Phoenix 준비됨 http://127.0.0.1:8694 (OTLP/HTTP http://127.0.0.1:8694/v1/traces)


In [48]:
GEN_URLS = {}
for name, style in [("gen-a", "base")]:      # 이미 있는 서버를 쓰려면 D69_GEN_URL_A, _B 에 주소를 넣습니다
    GEN_URLS[name] = os.environ.get(f"D69_GEN_URL_{name[-1].upper()}") or url(name)
    if GEN_URLS[name] == url(name):
        print(start(name, "gen_server", {"D69_NAME": name, "D69_GEN_MODEL": GEN_MODELS[name], "D69_FAKE_STYLE": style}))

gen-a 준비됨 http://127.0.0.1:8690


In [49]:
def start_rag(gen="gen-a"):
    stop("rag-server")
    return start("rag-server", "rag_server", {"D69_GEN_URL": GEN_URLS[gen], "D69_GEN_MODEL": GEN_MODELS[gen]})


print(start_rag("gen-a"))
print(start("gateway", "gateway", {"D69_RAG_URL": url("rag-server")}))
GW = url("gateway")

rag-server 준비됨 http://127.0.0.1:8692
gateway 준비됨 http://127.0.0.1:8693


In [50]:
from datetime import datetime


def ask(user, question, traceparent=None):
    """게이트웨이에 묻습니다. 실패해도 예외 대신 사유를 담은 dict를 반환합니다"""
    try:
        r = httpx.post(GW + "/v1/ask", json={"user": user, "question": question}, timeout=300,
                       headers={"traceparent": traceparent} if traceparent else {})
        return {**r.json(), "status": r.status_code, "trace_id": r.headers.get("x-request-id")}
    except (httpx.HTTPError, ValueError) as e:
        return {"status": None, "trace_id": None, "answer": "", "error": type(e).__name__}


def load_spans():
    ts = lambda t: datetime.fromisoformat(t.replace("Z", "+00:00")).timestamp()
    rows = []
    for path in sorted((WORK / "spans").glob("*.jsonl")):
        for s in map(json.loads, filter(str.strip, path.read_text(encoding="utf-8").splitlines())):
            rows.append({"trace_id": s["context"]["trace_id"][2:], "span_id": s["context"]["span_id"][2:],
                         "parent_id": (s.get("parent_id") or "")[2:] or None, "name": s["name"],
                         "service": s["resource"]["attributes"]["service.name"],
                         "start": ts(s["start_time"]), "end": ts(s["end_time"]), "attrs": s.get("attributes") or {}})
    return rows


first = ask("김총무", QMAP["Q01"]["q"])
print(json.dumps(first, ensure_ascii=False, indent=1))
SPANS = load_spans()
print(f"스팬 {len(SPANS)}개, 트레이스 {len({s['trace_id'] for s in SPANS})}개", dict(Counter(s["service"] for s in SPANS)))

{
 "answer": "답 : 2일",
 "sources": [
  "공개-02",
  "공개-06",
  "공개-15"
 ],
 "top_score": 0.9205,
 "model": "Qwen/Qwen2.5-0.5B-Instruct",
 "finish": "stop",
 "prompt_version": "v3",
 "embedding": "intfloat/multilingual-e5-small",
 "output_tokens": 6,
 "status": 200,
 "trace_id": "fe54f5e5ccb4e89fe6d995511525481b"
}
스팬 114개, 트레이스 19개 {'gateway': 19, 'generation': 19, 'rag-server': 76}


## 3. Phoenix 데이터셋과 평가 실험 만들기

평가 세트를 Phoenix에 등록하고, 각 입력을 서비스에 보내는 태스크와 위의 세 평가 함수를 연결합니다. 데이터셋의 참조 답은 서비스에 전달하지 않습니다. 자동 재시도와 반복 평가는 끄고 각 사례를 한 번 실행합니다.

In [51]:
from phoenix.client import Client
PX=Client(base_url=PHX)
RUN_TAG=str(time.time_ns())
DATASET=PX.datasets.create_dataset(
    name="d69-golden-"+RUN_TAG,
    inputs=[{"user":x["user"],"question":x["q"]} for x in EVAL_CASES],
    outputs=[{"answer":x["answer"],"sources":x["근거"]} for x in EVAL_CASES],
    metadata=[{"case_id":x["id"],"kind":x["kind"]} for x in EVAL_CASES],
)
ROWS=[]
def task(input,metadata):
    result=ask(input["user"],input["question"])
    ROWS.append({"case_id":metadata["case_id"],"question":input["question"],**result})
    return result

BASE_EXPERIMENT=PX.experiments.run_experiment(
    dataset=DATASET,task=task,
    evaluators={"answer_correct":answer_correct,"retrieval_hit":retrieval_hit,"citation_present":citation_present},
    experiment_name="baseline",experiment_metadata={"model":GEN_MODELS["gen-a"],"prompt":"v3","mode":"real-cpu-or-approved-endpoint"},
    repetitions=1,retries=0,timeout=180,
)
BASE_ROWS=list(ROWS)
(WORK/"baseline_responses.json").write_text(json.dumps(BASE_ROWS,ensure_ascii=False,indent=2),encoding="utf-8")
display(pd.DataFrame(BASE_ROWS)[["case_id","question","answer","sources","trace_id"]])

🧪 Experiment started.
📺 View dataset experiments: http://127.0.0.1:8694/datasets/RGF0YXNldDoy/experiments
🔗 View this experiment: http://127.0.0.1:8694/datasets/RGF0YXNldDoy/compare?experimentId=RXhwZXJpbWVudDoz


running tasks |██████████| 8/8 (100.0%) | ⏳ 04:01<00:00 | 30.16s/it


✅ Task runs completed.
🧠 Evaluation started.


running experiment evaluations |██████████| 24/24 (100.0%) | ⏳ 00:03<00:00 |  6.15it/s


Experiment completed: 8 task runs, 3 evaluator runs, 24 evaluations


,case_id,question,answer,sources,trace_id
0,Q01,재택근무는 한 주에 최대 며칠까지 신청할 수 있습니까?,답 : 2일,"[공개-02, 공개-06, 공개-15]",b3df5105c35b35791ec7be0ab645aa72
1,Q02,재택근무 신청은 전날 오후 몇 시까지 해야 합니까?,답 : 3\n\n근거 : 공개-02,"[공개-02, 공개-01, 공개-04]",2715c1db64c40b4f5602670d10b5cc39
2,Q05,국내 출장의 1박 숙박비는 최대 몇 만 원입니까?,답 : 10\n\n근거 : [공개-05] 문서 ID,"[공개-05, 공개-06, 공개-08]",17d0f1d5e32636b29ae7b252dc5ed7b5
3,Q07,보안 사고를 알게 되면 몇 시간 안에 알려야 합니까?,답 : 1시간,"[공개-07, 공개-14, 공개-02]",7a6442dfc68aa8f3027884e86263fb45
4,Q14,국내 출장을 3박 4일로 다녀오면 숙박비 상한과 식비를 합해 최대 몇 만 원을 받습니까?,답: 27\n\n근거: [공개-05]과 [공개-06] 두 문서 모두 숙박비와 식비의...,"[공개-05, 공개-06, 공개-08]",7cfdffc1e8749e9221ab61aa521910c5
5,Q18,S 등급은 팀별 상위 몇 퍼센트입니까?,답 : 10%,"[인사-01, 인사-02, 공개-17]",59eed7c7458ff09958fcbce642425904
6,Q19,S 등급은 팀별 상위 몇 퍼센트입니까?,답 : 30%,"[공개-17, 공개-16, 공개-10]",9ed48820e72313738abb527feecad39f
7,Q21,법인카드의 월 한도는 몇 만 원입니까?,답 : 50\n\n근거 : 공개-11,"[공개-11, 공개-16, 공개-06]",04b57246c4110ade58c1682430554e79


### Phoenix UI에서 사례별 결과 확인

로컬 브라우저에서 `http://localhost:8694`를 엽니다. **Datasets**에서 방금 출력한 `d69-golden-…`를 선택하고 **Experiments**의 `baseline`을 엽니다. 입력·참조 답·실제 답·세 평가 결과를 함께 읽습니다. 점수 하나만 보고 통과를 선언하지 말고, 실패한 답 한 건의 원문을 문서와 대조합니다. 참고 화면의 점수와 자신의 점수가 같을 필요는 없습니다.

In [61]:
from IPython.display import HTML,display
display(HTML('<a href="http://localhost:8694/datasets" target="_blank" rel="noopener">Phoenix 데이터셋 열기</a>'))
print("현재 데이터셋:","d69-golden-"+RUN_TAG)
REVIEW = {
    "사례 ID": "Q14",
    "검색 근거가 충분한가": "정답 문서인 공개-05(국내 출장비)가 검색되어 근거 자체는 충분하지만, 유사한 공개-06(해외 출장비)과 공개-08까지 함께 Top-3로 전달되어 문맥이 섞였다.",
    "답변의 오류": "숙박비(10만×3박)와 식비(3만×4일)를 합한 42 대신 27이라는 오답을 냈고, 국내 출장 질문에 해외 출장(공개-06)까지 근거로 섞어 설명하다 64토큰 제한에 걸려 문장이 중간('1박 20')에 끊겼다.",
    "자동 평가와 사람 판단의 차이": "자동 평가는 공개-05가 포함되었다는 이유만으로 retrieval_hit과 citation_present에 모두 1.0점을 주지만, 사람 눈으로 보면 오답을 유발한 불필요한 문서(공개-06) 인용과 답변 잘림까지 치명적 결함으로 보인다."
}

현재 데이터셋: d69-golden-1795760850629348137


## 4. 프롬프트를 고치고 같은 평가 다시 실행

검색 문서와 평가 세트를 유지하고 프롬프트만 바꿉니다. 아래 예시의 핵심은 제공 문서에서 확인한 숫자와 문서 ID를 함께 쓰고 근거가 없으면 답을 만들지 않게 하는 것입니다. 생성 모델을 다시 로드하지 않고 RAG 서버만 갱신합니다.

In [53]:
CANDIDATE_PROMPT=("당신은 사내 규정 안내 도우미입니다. 아래 제공된 문서만 사용합니다. "
    "질문에 답하는 숫자와 단위를 확인한 뒤 '답 : 숫자'를 첫 줄에 씁니다. "
    "다음 줄은 '근거 : 문서ID'로 쓰세요. 문서에 답이 없으면 '근거를 찾지 못했습니다.'라고만 답하세요. 한국어로 답하세요.")
os.environ["D69_SYSTEM_PROMPT"]=CANDIDATE_PROMPT
print(start_rag())
ROWS=[]
CANDIDATE_EXPERIMENT=PX.experiments.run_experiment(
    dataset=DATASET,task=task,
    evaluators={"answer_correct":answer_correct,"retrieval_hit":retrieval_hit,"citation_present":citation_present},
    experiment_name="prompt-candidate",experiment_metadata={"model":GEN_MODELS["gen-a"],"prompt":"candidate","mode":"real-cpu-or-approved-endpoint"},
    repetitions=1,retries=0,timeout=180,
)
CANDIDATE_ROWS=list(ROWS)
(WORK/"candidate_responses.json").write_text(json.dumps(CANDIDATE_ROWS,ensure_ascii=False,indent=2),encoding="utf-8")
print("같은 평가 세트로 실행:",len(BASE_ROWS),"→",len(CANDIDATE_ROWS))

rag-server 준비됨 http://127.0.0.1:8692
🧪 Experiment started.
📺 View dataset experiments: http://127.0.0.1:8694/datasets/RGF0YXNldDoy/experiments
🔗 View this experiment: http://127.0.0.1:8694/datasets/RGF0YXNldDoy/compare?experimentId=RXhwZXJpbWVudDo0


running tasks |██████████| 8/8 (100.0%) | ⏳ 03:58<00:00 | 29.85s/it


✅ Task runs completed.
🧠 Evaluation started.


running experiment evaluations |██████████| 24/24 (100.0%) | ⏳ 00:43<00:00 |  1.82s/it

Experiment completed: 8 task runs, 3 evaluator runs, 24 evaluations
같은 평가 세트로 실행: 8 → 8


![공식자료 또는 실제 실행 화면](lms-img:0)

실제 실행: 같은 8개 사례의 baseline·prompt-candidate 평가 실험.

Phoenix의 같은 데이터셋에서 두 실험을 선택해 비교합니다. 개선된 사례와 그대로인 사례, 나빠진 사례를 하나씩 찾습니다. 프롬프트를 바꿨다고 반드시 개선되는 것은 아닙니다. 검색된 근거 자체가 부족한 사례는 프롬프트 수정만으로 해결됐다고 기록하지 않습니다.

## 5. 실패 요청의 트레이스와 회귀 사례 확인

기준 평가에서 실패했거나 사람이 다시 검토해야 한다고 판단한 사례를 고릅니다. trace ID로 검색·생성 단계를 찾아, 잘못된 문서가 검색됐는지와 문서는 맞는데 생성이 틀렸는지를 구분합니다.

In [54]:
case_by_id={x["id"]:x for x in EVAL_CASES}
selected=next((r for r in BASE_ROWS if not grade(r.get("answer",""),case_by_id[r["case_id"]]["answer"])),BASE_ROWS[0])
print("사례:",selected["case_id"],selected["question"])
print("실제 답:",selected.get("answer"))
print("검색 출처:",selected.get("sources"))
print('Phoenix Spans 필터: trace_id == "'+str(selected["trace_id"])+'"')
first=selected

사례: Q14 국내 출장을 3박 4일로 다녀오면 숙박비 상한과 식비를 합해 최대 몇 만 원을 받습니까?
실제 답: 답: 27

근거: [공개-05]과 [공개-06] 두 문서 모두 숙박비와 식비의 최대 금액을 명시하고 있습니다. 각각 1박 10만원과 1박 20
검색 출처: ['공개-05', '공개-06', '공개-08']
Phoenix Spans 필터: trace_id == "7cfdffc1e8749e9221ab61aa521910c5"


![공식자료 또는 실제 실행 화면](lms-img:0)

실제 실행: RAG 요청의 게이트웨이·검색·생성 스팬을 연결한 Phoenix 트레이스.

In [55]:
# 실습 : 부모 ID -> 자식 스팬 목록인 children dict를 만드세요

def print_tree(trace_id, spans=None):
    # 동일한 요청의 스팬만 모아 서로 다른 요청의 검색과 생성을 연결하는 오류를 막습니다.
    spans = [s for s in (spans or load_spans()) if s["trace_id"] == trace_id]
    # 1) 부모 ID -> 자식 스팬 목록을 담을 defaultdict(list)
    children = defaultdict(list)
    # 2) 스팬마다 자기 parent_id 항목에 넣습니다
    for s in spans:
        children[s["parent_id"]].append(s)
    if not spans or children is None:
        print("스팬이 없거나 children 빈칸이 비어 있습니다")
        return
    t0, ids = min(s["start"] for s in spans), {s["span_id"] for s in spans}

    def walk(s, depth):
        print(f"{'   ' * depth}{s['name']:<44} [{s['service']:<10}] +{(s['start'] - t0) * 1000:7.1f}ms {(s['end'] - s['start']) * 1000:8.1f}ms")
        for c in sorted(children.get(s["span_id"], []), key=lambda x: x["start"]):
            walk(c, depth + 1)

    for root in sorted([s for s in spans if s["parent_id"] not in ids], key=lambda x: x["start"]):
        walk(root, 0)


print_tree(first["trace_id"])

POST /v1/ask                                 [gateway   ] +    0.0ms  50418.0ms
   POST /ask                                    [rag-server] +   11.0ms  50356.7ms
      embeddings intfloat/multilingual-e5-small    [rag-server] +   11.2ms   4473.0ms
      retrieval hangyeol-rules                     [rag-server] + 4494.0ms     51.5ms
      chat Qwen/Qwen2.5-0.5B-Instruct              [rag-server] + 4547.9ms  45812.1ms
         POST /v1/chat/completions                    [generation] + 4590.5ms  45689.2ms


### Phoenix 트레이스 화면에서 확인

**Tracing → default → Spans**에서 기본 `parent_id is None` 조건을 지우고 출력한 trace ID 필터를 적용합니다. 스팬을 열어 검색·생성의 부모·자식 관계를 읽고 **Attributes**의 모델명과 토큰 속성을 확인합니다. 질문·답변의 옳고 그름은 평가 사례와 원문으로 확인하고 트레이스는 원인 조사에 사용합니다.

In [60]:
REGRESSION_CASE={"case_id":selected["case_id"],"question":selected["question"],
    "expected":case_by_id[selected["case_id"]]["answer"],"sources":case_by_id[selected["case_id"]]["근거"],
    "observed":selected.get("answer"),"trace_id":selected["trace_id"]}
(WORK/"regression_case.json").write_text(json.dumps(REGRESSION_CASE,ensure_ascii=False,indent=2),encoding="utf-8")
print("다음 평가에 보관할 사례:",REGRESSION_CASE["case_id"])

다음 평가에 보관할 사례: Q14


## 추가 실험과 유의사항

- 운영 질문 두 기간의 기록이 있을 때 부서·언어·길이 분포를 비교해 새 사례를 평가 세트에 추가할 수 있습니다. 분포 변화만으로 모델 품질 저하의 원인을 확정하지 않습니다.
- 임베딩 평균 거리·TVD 계산과 Judge 편향 보정은 별도의 진단 주제입니다. 오늘의 필수 수행은 평가 세트 작성, 평가 실행, 원문 검토, 같은 세트 재평가와 실패 추적입니다.
- 소형 CPU 모델의 결과를 대형 모델의 품질이나 GPU 서빙 속도로 일반화하지 않습니다. 외부 모델로 바꿀 때는 모델·프롬프트·문서·생성 설정을 함께 기록합니다.

## 결과 정리와 제출

직접 실행한 요청과 응답, 생성한 파일을 근거로 오늘 구현한 기능과 남은 한계를 설명합니다. 실행하지 않은 조건은 미확인으로 기록합니다.

### 관련 개념 그림

아래 그림은 구현한 기능과 확장할 때 고려할 개념을 설명합니다.

![요청별 trace ID와 게이트웨이·검색·생성 스팬의 연결](lms-img:0)

*요청별 trace ID와 게이트웨이·검색·생성 스팬의 연결*

![기준 기간의 자연 변동과 현재 분포 변화를 비교하는 원리](lms-img:0)

*기준 기간의 자연 변동과 현재 분포 변화를 비교하는 원리*

![평가 모델을 변경할 때 같은 보정 세트로 판정을 비교하는 원리](lms-img:0)

*평가 모델을 변경할 때 같은 보정 세트로 판정을 비교하는 원리*

In [62]:
DECISION = "OpenTelemetry 분산 추적 워터폴(print_tree)과 Arize Phoenix 기반 RAG 평가(answer_correct, retrieval_hit, citation_present) 및 프롬프트 비교 실험을 구현해, 단순 조회(Q02, Q05)는 통과했으나 Q14의 계산 오류(27)·토큰 잘림·생성 지연(약 45초)과 Q19/Q21의 거절 실패(환각) 및 Q01 등의 근거 누락을 확인했으며, 0.5B 소형 모델의 추론·지시이행 한계와 규칙 기반 grade() 평가가 의미 전체를 검증하지 못하는 한계가 남아 있다."

if not DECISION:
    print("구현한 기능, 확인한 요청·응답과 남은 한계를 한 문장으로 작성하세요.")
else:
    save_submission(DECISION, baseline=BASE_ROWS, candidate=CANDIDATE_ROWS, review=REVIEW, regression=REGRESSION_CASE)

제출 파일: /workspace/work_d69/d69_submission.json


노트북과 제출 JSON을 저장한 뒤 서버를 종료합니다. 로컬 Docker를 사용했다면 호스트 터미널의 해당 `local_d69` 폴더에서 `docker compose down`을 실행합니다.

In [63]:
close_services()

실습 서버 종료 처리를 마쳤습니다.


## 개념 그림